# M06 · Théorie de l'information — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m06_information/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m06_information/03_notebook.ipynb`, créée par `python tools/start_chapter.py M06`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M06.1 à M06.9, M06.12, M06.13 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M06.16 | Entropies conjointe et conditionnelle d'un tableau · `joint_entropy, conditional_entropy` | 🔨 | ★ | 20 |
|  | M06.17 | L'information mutuelle · `mutual_information` | 🔨 | ★ | 15 |
|  | M06.18 | Codes préfixes et somme de Kraft · `kraft_sum, is_prefix_code` | 🔨 | ★ | 15 |
|  | M06.19 | L'écart de Jensen d'une loi discrète · `jensen_gap` | 🔨 | ★ | 15 |
|  | M06.20 | `sklearn.metrics.mutual_info_score` contre ta fonction, en nats | 📦 | ★ | 15 |
| B | M06.21 | Prédire : l'information mutuelle de X et 2X, de X et X² | 🔮 | ★ | 10 |
|  | M06.22 | Estimer une entropie à partir d'un échantillon : le biais des petits échantillons | 🔬 | ★★ | 20 |
|  | M06.23 | Un diagramme de l'information : H(X), H(Y), H(X \| Y), I(X ; Y) | 🎨 | ★ | 20 |
| C | M06.24 | Chasse à l'erreur : log en base 10, 0 log 0, KL à l'envers | 🐛 | ★ | 15 |
|  | M06.25 | Calculer 0 log 0 proprement (`scipy.special.xlogy`) | 🛠️ | ★ | 10 |
|  | M06.26 | Défi : deviner un nombre de 1 à 100 en un minimum de questions | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M06")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : information et entropie, entropies conjointe et conditionnelle, information mutuelle, cross-entropy et KL, inégalité de Jensen, Kraft, longueur moyenne d'un code, log loss d'un classifieur, choix d'une feature.
- Écrire les six fonctions de `mymath/info.py` et les tester, puis utiliser scikit-learn.
- Prédire l'information mutuelle de $X$ avec $2X$ et $X^2$, mesurer le biais de l'entropie estimée, dessiner le diagramme de l'information, chasser des erreurs, calculer $0 \log 0$ proprement, deviner un nombre en un minimum de questions.

**Rappel express.** $H(X) = -\sum p \log_2 p$ ; $H(X, Y) = H(X) + H(Y \mid X)$ ; $I(X ; Y) = H(Y) - H(Y \mid X) = H(X) + H(Y) - H(X, Y) \ge 0$ ; $H(p, q) = -\sum p \log_2 q = H(p) + \mathrm{KL}(p \,\|\, q)$ ; Jensen : $\mathbb{E}[f(X)] \ge f(\mathbb{E}[X])$ pour $f$ convexe ; Kraft : $\sum 2^{-\ell_i} \le 1$ ; le meilleur code préfixe vérifie $H \le \bar{L} < H + 1$ ; 1 nat $\approx 1{,}443$ bit.

## Partie 0 · Vérifier tes exercices papier (✏️ M06.1 à M06.9, 🔗 M06.12 et M06.13)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une liste `[-1.5, 2.5]`, `True` / `False`, ou une lettre ou un nom entre guillemets, `"B"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M06.1 — L'information d'événements et l'entropie d'une loi à quatre issues**

In [ ]:
# Ex M06.1: the values you found on paper
answer_M06_1a = ...  # a) an integer
answer_M06_1b = ...  # b) an integer
answer_M06_1c = ...  # c) a number (3 decimals)
answer_M06_1d = ...  # d) a number (3 decimals)
answer_M06_1e = ...  # e) an integer
answer_M06_1f = ...  # f) a number (3 decimals)
answer_M06_1g = ...  # g) a number (3 decimals)
answer_M06_1h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M06_1a, answer_M06_1b, answer_M06_1c, answer_M06_1d, answer_M06_1e, answer_M06_1f, answer_M06_1g, answer_M06_1h]):
    wbm.check(f"M06.1{letter}", answer)

**Ex M06.2 — L'entropie maximale : comparer trois lois à trois issues**

In [ ]:
# Ex M06.2: the values you found on paper
answer_M06_2a = ...  # a) a number (3 decimals)
answer_M06_2b = ...  # b) an integer
answer_M06_2c = ...  # c) a number (3 decimals)
answer_M06_2d = ...  # d) a letter, as text
answer_M06_2e = ...  # e) a number (3 decimals)
answer_M06_2f = ...  # f) a number (3 decimals)
answer_M06_2g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M06_2a, answer_M06_2b, answer_M06_2c, answer_M06_2d, answer_M06_2e, answer_M06_2f, answer_M06_2g]):
    wbm.check(f"M06.2{letter}", answer)

**Ex M06.3 — Entropies conjointe et conditionnelle d'un tableau 2 × 3**

In [ ]:
# Ex M06.3: the values you found on paper
answer_M06_3a = ...  # a) a number (2 decimals)
answer_M06_3b = ...  # b) a number (1 decimal)
answer_M06_3c = ...  # c) a number (2 decimals)
answer_M06_3d = ...  # d) a number (3 decimals)
answer_M06_3e = ...  # e) a number (3 decimals)
answer_M06_3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_M06_3a, answer_M06_3b, answer_M06_3c, answer_M06_3d, answer_M06_3e, answer_M06_3f]):
    wbm.check(f"M06.3{letter}", answer)

**Ex M06.4 — La règle de la chaîne de l'entropie, vérifiée**

In [ ]:
# Ex M06.4: the values you found on paper
answer_M06_4a = ...  # a) an integer
answer_M06_4b = ...  # b) a number (3 decimals)
answer_M06_4c = ...  # c) a number (3 decimals)
answer_M06_4d = ...  # d) a number (3 decimals)
answer_M06_4e = ...  # e) a number (3 decimals)
answer_M06_4f = ...  # f) a number (3 decimals)
answer_M06_4g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M06_4a, answer_M06_4b, answer_M06_4c, answer_M06_4d, answer_M06_4e, answer_M06_4f, answer_M06_4g]):
    wbm.check(f"M06.4{letter}", answer)

**Ex M06.5 — L'information mutuelle de deux variables**

In [ ]:
# Ex M06.5: the values you found on paper
answer_M06_5a = ...  # a) an irreducible fraction, as text
answer_M06_5b = ...  # b) a number (3 decimals)
answer_M06_5c = ...  # c) a number (3 decimals)
answer_M06_5d = ...  # d) a number (3 decimals)
answer_M06_5e = ...  # e) a number (3 decimals)
answer_M06_5f = ...  # f) a number (1 decimal)
answer_M06_5g = ...  # g) a number (3 decimals)
answer_M06_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M06_5a, answer_M06_5b, answer_M06_5c, answer_M06_5d, answer_M06_5e, answer_M06_5f, answer_M06_5g, answer_M06_5h]):
    wbm.check(f"M06.5{letter}", answer)

**Ex M06.6 — Cross-entropy et KL en bits perdus : un code fait pour la mauvaise loi**

In [ ]:
# Ex M06.6: the values you found on paper
answer_M06_6a = ...  # a) a number (2 decimals)
answer_M06_6b = ...  # b) an integer
answer_M06_6c = ...  # c) a number (1 decimal)
answer_M06_6d = ...  # d) a number (2 decimals)
answer_M06_6e = ...  # e) an integer
answer_M06_6f = ...  # f) a number (4 decimals)
answer_M06_6g = ...  # g) a number (2 decimals)
answer_M06_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M06_6a, answer_M06_6b, answer_M06_6c, answer_M06_6d, answer_M06_6e, answer_M06_6f, answer_M06_6g, answer_M06_6h]):
    wbm.check(f"M06.6{letter}", answer)

**Ex M06.7 — L'inégalité de Jensen sur trois exemples**

In [ ]:
# Ex M06.7: the values you found on paper
answer_M06_7a = ...  # a) an integer
answer_M06_7b = ...  # b) an integer
answer_M06_7c = ...  # c) an integer
answer_M06_7d = ...  # d) a number (4 decimals)
answer_M06_7e = ...  # e) a number (1 decimal)
answer_M06_7f = ...  # f) a number (4 decimals)
answer_M06_7g = ...  # g) a number (4 decimals)
answer_M06_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M06_7a, answer_M06_7b, answer_M06_7c, answer_M06_7d, answer_M06_7e, answer_M06_7f, answer_M06_7g, answer_M06_7h]):
    wbm.check(f"M06.7{letter}", answer)

**Ex M06.8 — Kraft : ces longueurs de code sont-elles possibles ?**

In [ ]:
# Ex M06.8: the values you found on paper
answer_M06_8a = ...  # a) an irreducible fraction, as text
answer_M06_8b = ...  # b) an irreducible fraction, as text
answer_M06_8c = ...  # c) an irreducible fraction, as text
answer_M06_8d = ...  # d) an integer
answer_M06_8e = ...  # e) an integer
answer_M06_8f = ...  # f) an integer
answer_M06_8g = ...  # g) True or False
answer_M06_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M06_8a, answer_M06_8b, answer_M06_8c, answer_M06_8d, answer_M06_8e, answer_M06_8f, answer_M06_8g, answer_M06_8h]):
    wbm.check(f"M06.8{letter}", answer)

**Ex M06.9 — La longueur moyenne d'un code, comparée à l'entropie**

In [ ]:
# Ex M06.9: the values you found on paper
answer_M06_9a = ...  # a) a number (3 decimals)
answer_M06_9b = ...  # b) a number (1 decimal)
answer_M06_9c = ...  # c) an integer
answer_M06_9d = ...  # d) a list of 5 integers
answer_M06_9e = ...  # e) an irreducible fraction, as text
answer_M06_9f = ...  # f) a number (1 decimal)
answer_M06_9g = ...  # g) True or False
answer_M06_9h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_M06_9a, answer_M06_9b, answer_M06_9c, answer_M06_9d, answer_M06_9e, answer_M06_9f, answer_M06_9g, answer_M06_9h]):
    wbm.check(f"M06.9{letter}", answer)

**Ex M06.12 — La log loss d'un classifieur, une cross-entropy moyenne**

In [ ]:
# Ex M06.12: the values you found on paper
answer_M06_12a = ...  # a) a number (4 decimals)
answer_M06_12b = ...  # b) a number (4 decimals)
answer_M06_12c = ...  # c) a number (4 decimals)
answer_M06_12d = ...  # d) a number (4 decimals)
answer_M06_12e = ...  # e) True or False
answer_M06_12f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M06_12a, answer_M06_12b, answer_M06_12c, answer_M06_12d, answer_M06_12e, answer_M06_12f]):
    wbm.check(f"M06.12{letter}", answer)

**Ex M06.13 — Choisir une feature par l'information mutuelle**

In [ ]:
# Ex M06.13: the values you found on paper
answer_M06_13a = ...  # a) a number (4 decimals)
answer_M06_13b = ...  # b) a number (4 decimals)
answer_M06_13c = ...  # c) a number (4 decimals)
answer_M06_13d = ...  # d) a name, as text
answer_M06_13e = ...  # e) a number (4 decimals)
answer_M06_13f = ...  # f) a number (4 decimals)
answer_M06_13g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M06_13a, answer_M06_13b, answer_M06_13c, answer_M06_13d, answer_M06_13e, answer_M06_13f, answer_M06_13g]):
    wbm.check(f"M06.13{letter}", answer)

## Partie A · Ta librairie : entropies d'un tableau, information mutuelle, Kraft, Jensen ; scikit-learn

Fiche §3, §4, §6 et §7. Les six fonctions de `mymath/info.py`, puis l'information mutuelle de scikit-learn.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M06.16 — Entropies conjointe et conditionnelle d'un tableau · `joint_entropy, conditional_entropy` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer l'entropie conjointe et l'entropie conditionnelle d'une loi jointe, avec la convention 0 log 0 = 0, dans une base au choix, en refusant ce qui n'est pas une loi.  
**Prérequis :** fiche §3 ; M03 (`conditional.py`) · **mymath :** `info.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/info.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Une loi jointe est un tableau à deux dimensions (une liste de listes ou un tableau NumPy), comme dans `conditional.py` (M03) : les lignes sont les valeurs de $X$, les colonnes celles de $Y$. NumPy suffit ; `scipy.stats.entropy` et scikit-learn sont interdits : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Les nombres renvoyés sont des `float` de Python, les réponses oui ou non des `bool` de Python.

Écris (fiche §3) :
- `joint_entropy(joint, base=2)` : $H(X, Y) = -\sum_{x, y} p(x, y) \log p(x, y)$, sur les seules cases strictement positives ;
- `conditional_entropy(joint, base=2)` : $H(Y \mid X)$, l'entropie de la variable des **colonnes** quand on connaît celle des **lignes** (par la règle de la chaîne, $H(X, Y) - H(X)$, où la loi de $X$ est faite des sommes des lignes ; ou par la double somme).

Les deux lèvent une `ValueError` si le tableau n'est pas une loi jointe (pas deux dimensions, une case négative ou `nan`, une somme différente de 1 à $10^{-9}$ près), ou si `base` n'est pas plus grande que 1.

La vérification calcule, avec `T16 = [[0.2, 0.1, 0.1], [0.05, 0.25, 0.3]]` :
- a) `joint_entropy(T16)` (4 décimales) ;
- b) `conditional_entropy(T16)` (4 décimales) ;
- c) `conditional_entropy(np.array(T16).T)` : le tableau transposé (4 décimales) ;
- d) `joint_entropy(T16, base=math.e)` (4 décimales) ;
- e) le nom de l'erreur levée par `joint_entropy([[0.5, 0.4], [0.2, -0.1]])` ;
- f) le nom de l'erreur levée par `conditional_entropy(T16, base=1)` ;
- g) `joint_entropy([[0.15, 0], [0, 0.85]])` (4 décimales) ;
- h) le nom du type de `conditional_entropy(T16)`, en texte.

Dans ta copie : quelle entropie conditionnelle c) calcule-t-elle, et pourquoi la transposition suffit-elle ? Vérifie b) à la main, ligne par ligne. Pourquoi le tableau de e) est-il refusé alors que sa somme vaut 1 ? À quelle entropie d'une loi à deux issues g) est-elle égale ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M06")  # reload your saved file

In [ ]:
T16 = [[0.2, 0.1, 0.1], [0.05, 0.25, 0.3]]
with checked("M06.16a"):
    wbm.check("M06.16a", mymath.info.joint_entropy(T16), computed=True)
with checked("M06.16b"):
    wbm.check("M06.16b", mymath.info.conditional_entropy(T16), computed=True)
with checked("M06.16c"):
    wbm.check("M06.16c", mymath.info.conditional_entropy(np.array(T16).T), computed=True)
with checked("M06.16d"):
    wbm.check("M06.16d", mymath.info.joint_entropy(T16, base=math.e), computed=True)
with checked("M06.16e"):
    wbm.check("M06.16e", raised(mymath.info.joint_entropy, [[0.5, 0.4], [0.2, -0.1]]), computed=True)
with checked("M06.16f"):
    wbm.check("M06.16f", raised(mymath.info.conditional_entropy, T16, base=1), computed=True)
with checked("M06.16g"):
    wbm.check("M06.16g", mymath.info.joint_entropy([[0.15, 0], [0, 0.85]]), computed=True)
with checked("M06.16h"):
    wbm.check("M06.16h", type(mymath.info.conditional_entropy(T16)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m06_info.py", "test_joint_entropy_ or test_conditional_entropy_ or test_info_uses_")

### Ex M06.17 — L'information mutuelle · `mutual_information` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer l'information mutuelle d'une loi jointe, et vérifier qu'elle est symétrique, nulle pour des variables indépendantes et égale à l'entropie quand une variable détermine l'autre.  
**Prérequis :** fiche §4 ; Ex M06.16 · **mymath :** `info.py` · **Parcours :** C, E

Écris `mutual_information(joint, base=2)` (fiche §4) : $I(X ; Y) = \sum p(x, y) \log \frac{p(x, y)}{p(x)\,p(y)}$ sur les cases strictement positives, ou $H(Y) - H(Y \mid X)$ avec ta fonction de M06.16 ; une petite valeur négative due aux arrondis (au-dessus de $-10^{-12}$) est renvoyée comme `0.0`. Mêmes `ValueError` qu'en M06.16.

La vérification calcule, avec `T17 = [[0.3, 0.1], [0.05, 0.25], [0.15, 0.15]]` :
- a) `mutual_information(T17)` (4 décimales) ;
- b) l'information mutuelle du tableau transposé est-elle égale à a), à $10^{-12}$ près ? (`True` ou `False`) ;
- c) `mutual_information(np.outer([0.2, 0.8], [0.3, 0.3, 0.4]))` (4 décimales) ;
- d) `mutual_information(np.diag([0.1, 0.4, 0.5]))` (4 décimales) ;
- e) `mutual_information(T17, base=math.e)` (4 décimales) ;
- f) le nom de l'erreur levée par `mutual_information([[0.5, np.nan], [0.25, 0.25]])` ;
- g) le nom du type de `mutual_information(T17)`, en texte.

Dans ta copie : quelle loi `np.outer` construit-elle en c), et pourquoi le résultat était-il prévisible ? Même question pour d) : que vaut $H(Y \mid X)$ pour ce tableau ? Pourquoi la vérification de f) ne peut-elle pas s'écrire `np.any(table < 0)` ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M06")  # reload your saved file

In [ ]:
T17 = [[0.3, 0.1], [0.05, 0.25], [0.15, 0.15]]
with checked("M06.17a"):
    wbm.check("M06.17a", mymath.info.mutual_information(T17), computed=True)
with checked("M06.17b"):
    wbm.check("M06.17b", math.isclose(mymath.info.mutual_information(T17),
                                      mymath.info.mutual_information(np.array(T17).T), abs_tol=1e-12), computed=True)
with checked("M06.17c"):
    wbm.check("M06.17c", mymath.info.mutual_information(np.outer([0.2, 0.8], [0.3, 0.3, 0.4])), computed=True)
with checked("M06.17d"):
    wbm.check("M06.17d", mymath.info.mutual_information(np.diag([0.1, 0.4, 0.5])), computed=True)
with checked("M06.17e"):
    wbm.check("M06.17e", mymath.info.mutual_information(T17, base=math.e), computed=True)
with checked("M06.17f"):
    wbm.check("M06.17f", raised(mymath.info.mutual_information, [[0.5, np.nan], [0.25, 0.25]]), computed=True)
with checked("M06.17g"):
    wbm.check("M06.17g", type(mymath.info.mutual_information(T17)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m06_info.py", "test_mutual_information_ or test_info_uses_")

### Ex M06.18 — Codes préfixes et somme de Kraft · `kraft_sum, is_prefix_code` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la somme de Kraft de longueurs de code pour un alphabet quelconque, et le test qui dit si un code est préfixe.  
**Prérequis :** fiche §7 · **mymath :** `info.py` · **Parcours :** C

Écris (fiche §7) :
- `kraft_sum(lengths, alphabet=2)` : $\sum_i D^{-\ell_i}$, où $D$ est le nombre de symboles de l'alphabet du code ; une `ValueError` si la liste est vide, si une longueur est plus petite que 1, ou si $D < 2$ ;
- `is_prefix_code(codewords)` : `True` si aucun mot n'est le début d'un autre (`str.startswith`), un mot répété deux fois rendant le code non préfixe ; une `ValueError` si un mot est le texte vide.

La vérification calcule :
- a) `kraft_sum([2, 2, 3, 4, 4, 4])` (4 décimales) ;
- b) `kraft_sum([1, 2, 2, 3], alphabet=3)` (4 décimales) ;
- c) le nom de l'erreur levée par `kraft_sum([2, 0, 1])` ;
- d) `is_prefix_code(["10", "110", "00", "111", "01"])` ;
- e) `is_prefix_code(["0", "10", "101", "11"])` ;
- f) `is_prefix_code(["1", "01", "1"])` ;
- g) le nom de l'erreur levée par `is_prefix_code(["0", ""])` ;
- h) `isinstance(is_prefix_code(np.array(["0", "10"])), bool)`.

Dans ta copie : pour a) et b), un code préfixe de ces longueurs existe-t-il ? Pour d), dessine l'arbre du code. Pourquoi le mot vide de g) ne peut-il pas faire partie d'un code (de quel mot est-il le début) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M06")  # reload your saved file

In [ ]:
with checked("M06.18a"):
    wbm.check("M06.18a", mymath.info.kraft_sum([2, 2, 3, 4, 4, 4]), computed=True)
with checked("M06.18b"):
    wbm.check("M06.18b", mymath.info.kraft_sum([1, 2, 2, 3], alphabet=3), computed=True)
with checked("M06.18c"):
    wbm.check("M06.18c", raised(mymath.info.kraft_sum, [2, 0, 1]), computed=True)
with checked("M06.18d"):
    wbm.check("M06.18d", mymath.info.is_prefix_code(["10", "110", "00", "111", "01"]), computed=True)
with checked("M06.18e"):
    wbm.check("M06.18e", mymath.info.is_prefix_code(["0", "10", "101", "11"]), computed=True)
with checked("M06.18f"):
    wbm.check("M06.18f", mymath.info.is_prefix_code(["1", "01", "1"]), computed=True)
with checked("M06.18g"):
    wbm.check("M06.18g", raised(mymath.info.is_prefix_code, ["0", ""]), computed=True)
with checked("M06.18h"):
    wbm.check("M06.18h", isinstance(mymath.info.is_prefix_code(np.array(["0", "10"])), bool), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m06_info.py", "test_kraft_sum_ or test_is_prefix_code_ or test_info_uses_")

### Ex M06.19 — L'écart de Jensen d'une loi discrète · `jensen_gap` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer l'écart de Jensen d'une variable discrète, avec des poids donnés ou égaux, et lire son signe.  
**Prérequis :** fiche §6 · **mymath :** `info.py` · **Parcours :** C

Écris `jensen_gap(f, values, weights=None)` (fiche §6) : $\mathbb{E}[f(X)] - f(\mathbb{E}[X])$ pour la variable $X$ qui prend les valeurs `values` avec les probabilités `weights` (toutes égales si `weights` vaut `None`). `f` est une fonction d'une variable réelle, comme `math.log` : applique-la valeur par valeur. Une `ValueError` si `values` est vide, ou si les poids n'ont pas la bonne longueur, ne sont pas positifs ou ne somment pas à 1 (à $10^{-9}$ près).

La vérification calcule :
- a) `jensen_gap(lambda x: x ** 2, [0, 2, 5], [0.5, 0.3, 0.2])` (4 décimales) ;
- b) `jensen_gap(math.log, [1, 2, 8])` (4 décimales) ;
- c) `jensen_gap(math.exp, [-1, 1])` (4 décimales) ;
- d) `jensen_gap(abs, [-2, 2])` (4 décimales) ;
- e) le nom de l'erreur levée par `jensen_gap(lambda x: x, [1, 2], [0.7, 0.2])` ;
- f) le nom de l'erreur levée par `jensen_gap(lambda x: x, [])` ;
- g) le nom du type de `jensen_gap(lambda x: x ** 2, np.array([1.0, 2.0]))`, en texte.

Dans ta copie : pour a) à d), dis si la fonction est convexe ou concave, et vérifie le signe. Calcule a) à la main : quelle grandeur de M02b retrouves-tu ? Pourquoi `abs`, convexe mais pas strictement convexe, donne-t-il un écart non nul en d), et pour quelles valeurs l'écart serait-il nul ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M06")  # reload your saved file

In [ ]:
with checked("M06.19a"):
    wbm.check("M06.19a", mymath.info.jensen_gap(lambda x: x ** 2, [0, 2, 5], [0.5, 0.3, 0.2]), computed=True)
with checked("M06.19b"):
    wbm.check("M06.19b", mymath.info.jensen_gap(math.log, [1, 2, 8]), computed=True)
with checked("M06.19c"):
    wbm.check("M06.19c", mymath.info.jensen_gap(math.exp, [-1, 1]), computed=True)
with checked("M06.19d"):
    wbm.check("M06.19d", mymath.info.jensen_gap(abs, [-2, 2]), computed=True)
with checked("M06.19e"):
    wbm.check("M06.19e", raised(mymath.info.jensen_gap, lambda x: x, [1, 2], [0.7, 0.2]), computed=True)
with checked("M06.19f"):
    wbm.check("M06.19f", raised(mymath.info.jensen_gap, lambda x: x, []), computed=True)
with checked("M06.19g"):
    wbm.check("M06.19g", type(mymath.info.jensen_gap(lambda x: x ** 2, np.array([1.0, 2.0]))).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m06_info.py", "test_jensen_gap_ or test_info_uses_")

### Ex M06.20 — `sklearn.metrics.mutual_info_score` contre ta fonction, en nats 📦 ★ ⏱️ 15 min
**Objectif :** calculer une information mutuelle avec scikit-learn à partir d'observations, en connaître l'unité et les propriétés, et voir son biais sur un petit échantillon.  
**Prérequis :** fiche §4 et §8 ; Ex M06.17 · **Parcours :** C

scikit-learn calcule l'information mutuelle de deux variables catégorielles à partir de leurs **observations** (deux tableaux de valeurs de même longueur, une case par observation), pas d'une loi jointe : `mutual_info_score(x, y)` compte les paires, divise par leur nombre, et renvoie le résultat en **nats**. La cellule donnée crée `x_20` et `y_20`, 40 observations de deux variables à trois valeurs chacune. Calcule :
- a) `mutual_info_score(x_20, y_20)` (4 décimales) ;
- b) ce résultat en bits (4 décimales) ;
- c) `mutual_info_score(x_20, x_20)` (4 décimales) ;
- d) en remplaçant les valeurs 0, 1, 2 de `x_20` par les noms `"rouge"`, `"vert"`, `"bleu"` (`np.array(["rouge", "vert", "bleu"])[x_20]`), le résultat de a) change-t-il ? (`True` ou `False`) ;
- e) `mutual_info_score(x_20, rng.permutation(y_20))`, avec `rng = np.random.default_rng(620)` : les observations de $Y$ mélangées au hasard, ce qui rend les deux variables indépendantes (4 décimales).

La cellule suivante compare ensuite scikit-learn et ta fonction `mutual_information` (M06.17), en nats, sur la loi jointe des comptages.

Dans ta copie : quelle entropie c) donne-t-elle ? Pourquoi e) n'est-elle pas nulle, alors que les variables mélangées sont indépendantes ? Que deviendrait-elle avec 4 000 observations ? Pourquoi est-ce un piège quand on classe des features par information mutuelle sur peu de données (M06.13, fiche §8) ?

In [ ]:
from sklearn.metrics import mutual_info_score

COUNTS_20 = np.array([[9, 3, 0], [2, 6, 4], [1, 3, 12]])     # 40 observations: rows X = 0, 1, 2; columns Y = 0, 1, 2
rows_20, cols_20 = np.indices(COUNTS_20.shape)
x_20 = np.repeat(rows_20.ravel(), COUNTS_20.ravel())         # the value of X of each observation
y_20 = np.repeat(cols_20.ravel(), COUNTS_20.ravel())         # the value of Y of the same observation
order_20 = np.random.default_rng(20).permutation(x_20.size)  # the observations in a random order
x_20, y_20 = x_20[order_20], y_20[order_20]
print("x_20[:12] =", x_20[:12], "· y_20[:12] =", y_20[:12])

In [ ]:
answer_M06_20a = ...  # a) a number (4 decimals)
answer_M06_20b = ...  # b) a number (4 decimals)
answer_M06_20c = ...  # c) a number (4 decimals)
answer_M06_20d = ...  # d) True or False
answer_M06_20e = ...  # e) a number (4 decimals)

In [ ]:
for letter, answer in zip("abcde", [answer_M06_20a, answer_M06_20b, answer_M06_20c, answer_M06_20d,
                                    answer_M06_20e]):
    wbm.check(f"M06.20{letter}", answer)

**Ta fonction contre scikit-learn** (cette cellule utilise ta fonction de M06.17).

In [ ]:
with checked("M06.20 comparaison"):
    law_20 = COUNTS_20 / COUNTS_20.sum()
    mine_20 = mymath.info.mutual_information(law_20, base=math.e)
    theirs_20 = mutual_info_score(x_20, y_20)
    print(f"mutual_information (nats): {mine_20:.6f} · mutual_info_score: {theirs_20:.6f}")
    if abs(mine_20 - theirs_20) < 1e-9:
        print("✅ Ex M06.20 : ta fonction et scikit-learn donnent la même information mutuelle.")
    else:
        print("❌ Ex M06.20 : ta fonction (en nats, base=math.e) et scikit-learn diffèrent : revois M06.17.")

## Partie B · Une prédiction, une expérience et une figure

Fiche §2, §3 et §4. L'information mutuelle d'une variable avec une fonction d'elle-même, le biais de l'entropie estimée sur un petit échantillon, le diagramme de l'information en barres.

### Ex M06.21 — Prédire : l'information mutuelle de X et 2X, de X et X² 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'information mutuelle d'une variable avec une fonction d'elle-même, et la comparer à la corrélation.  
**Prérequis :** fiche §4 · **Parcours :** C, E

Soit $X$ uniforme sur $\{-2, -1, 0, 1, 2\}$, d'entropie $H(X) = \log_2 5$. Choisis, pour chaque question :
- a) $I(X ; 2X)$ : (1) 0 ; (2) strictement entre 0 et $H(X)$ ; (3) égale à $H(X)$ ; (4) égale à $2\,H(X)$ ;
- b) $I(X ; X^2)$ : (1) 0, car la corrélation de $X$ et $X^2$ est nulle ; (2) égale à $H(X)$ ; (3) strictement entre 0 et $H(X)$ ; (4) plus grande que $H(X)$ ;
- c) la corrélation de $X$ et $X^2$ : (1) 0 ; (2) 1 ; (3) strictement entre 0 et 1 ; (4) négative ;
- d) $I(X ; X^2)$ est égale à : (1) $H(X) - H(X^2)$ ; (2) $H(X^2)$ ; (3) $H(X) + H(X^2)$ ; (4) $H(X \mid X^2)$.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience. Elle utilise les fonctions de la référence (les tiennes, M06.16 et M06.17, font la même chose).

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M06_21a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M06_21b = ...  # b) 1, 2, 3 or 4
prediction_M06_21c = ...  # c) 1, 2, 3 or 4
prediction_M06_21d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M06_21a, prediction_M06_21b, prediction_M06_21c,
                                      prediction_M06_21d]):
    wbm.check(f"M06.21{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M06_21a, prediction_M06_21b, prediction_M06_21c, prediction_M06_21d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    ref_21 = wbm.load_mymath("ref")                        # the reference library: the same results as yours
    values_21 = np.arange(-2, 3)                            # X is uniform on {-2, -1, 0, 1, 2}


    def joint_21(g):
        """The joint law of (X, g(X)): one row per value of X, one column per value of g(X)."""
        images = sorted({int(g(v)) for v in values_21})
        table = np.zeros((len(values_21), len(images)))
        for i, v in enumerate(values_21):
            table[i, images.index(int(g(v)))] = 1 / len(values_21)
        return table


    H_X_21 = ref_21.info.joint_entropy([np.full(5, 0.2)])  # the entropy of X: a table with a single row
    print(f"H(X) = {H_X_21:.4f} bits")
    for name, g in [("2X", lambda v: 2 * v), ("X²", lambda v: v ** 2)]:
        table = joint_21(g)
        r = round(float(np.corrcoef(values_21, [g(v) for v in values_21])[0, 1]), 10) + 0.0
        print(f"I(X ; {name}) = {ref_21.info.mutual_information(table):.4f} · "
              f"H({name}) = {ref_21.info.joint_entropy(table.sum(axis=0, keepdims=True)):.4f} · "
              f"H(X | {name}) = {ref_21.info.conditional_entropy(table.T):.4f} · correlation of X and {name}: {r:.4f}")

**Après l'expérience**, dans ta copie : pourquoi $I(X ; 2X) = H(X)$ (que vaut $H(X \mid 2X)$) ? Pourquoi $I(X ; X^2) = H(X^2)$ (que vaut $H(X^2 \mid X)$) ? Que reste-t-il d'inconnu sur $X$ quand on connaît $X^2$, et combien de bits cela fait-il ? Pourquoi la corrélation ne voit-elle rien, et quelle figure de la fiche le montre ?

### Ex M06.22 — Estimer une entropie à partir d'un échantillon : le biais des petits échantillons 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le biais de l'entropie estimée par les fréquences d'un échantillon, voir comment il diminue avec la taille de l'échantillon, et le corriger.  
**Prérequis :** fiche §2 ; M04 (estimation) · **Parcours :** C

On estime souvent une entropie à partir d'un échantillon : on remplace chaque probabilité par sa fréquence observée, puis on applique la formule. C'est l'estimateur « plug-in ». Il se trompe en moyenne, et toujours dans le même sens. Mesure ce biais.

Écris `plugin_entropy(sample, k)`, l'entropie en bits des fréquences de l'échantillon `sample`, un tableau d'entiers de 0 à `k − 1` (`np.bincount(sample, minlength=k)` compte les valeurs ; une fréquence nulle compte pour 0), en `float`.

La cellule donnée tire, pour la loi `P_22` (6 issues, d'entropie `H_22`), 2 000 échantillons de chaque taille $n$ = 10, 20, 50, 100 et 1 000 : `SAMPLES_22[n]` est un tableau de forme `(2000, n)`, un échantillon par ligne. La vérification calcule, avec ta fonction :
- a) la moyenne des 2 000 estimations pour $n = 10$ (3 décimales) ;
- b) le biais pour $n = 10$ : cette moyenne moins la vraie entropie `H_22` (3 décimales) ;
- c) le biais pour $n = 100$ (3 décimales) ;
- d) la proportion des 2 000 échantillons de taille 10 dont l'estimation est plus petite que `H_22` (2 décimales) ;
- e) le biais pour $n = 10$ après la correction de Miller-Madow, qui ajoute $\frac{k - 1}{2n\ln 2}$ à chaque estimation (3 décimales).

Une dernière cellule affiche le biais pour les cinq tailles.

Dans ta copie : dans quel sens l'estimateur plug-in se trompe-t-il, et pourquoi (que devient une issue rare, de probabilité 0,05, qui n'apparaît pas dans un échantillon de taille 10) ? Compare les biais b) et c) avec l'approximation $-\frac{k - 1}{2n\ln 2}$ : pour quelle taille l'approximation est-elle la meilleure ? Qu'en conclure pour une information mutuelle estimée sur peu de données (M06.20 e)) ?

In [ ]:
P_22 = np.array([0.3, 0.25, 0.2, 0.12, 0.08, 0.05])     # a law on 6 outcomes, 0 to 5
H_22 = float(-np.sum(P_22 * np.log2(P_22)))                 # its entropy, in bits
rng_22 = np.random.default_rng(622)
SAMPLES_22 = {n: rng_22.choice(6, size=(2000, n), p=P_22) for n in (10, 20, 50, 100, 1000)}   # 2000 samples of each size
print(f"H = {H_22:.4f} bits ·", {n: samples.shape for n, samples in SAMPLES_22.items()})

In [ ]:
def plugin_entropy(sample, k):
    """The entropy, in bits, of the frequencies of sample, an array of integers from 0 to k - 1 (0 log 0 = 0)."""
    raise NotImplementedError("np.bincount(sample, minlength=k), les fréquences, les seules non nulles, puis la somme")

In [ ]:
def mean_estimate_22(n):
    """The mean of the plug-in estimates of the 2000 samples of size n."""
    return float(np.mean([plugin_entropy(sample, 6) for sample in SAMPLES_22[n]]))


with checked("M06.22a"):
    wbm.check("M06.22a", mean_estimate_22(10), computed=True)
with checked("M06.22b"):
    wbm.check("M06.22b", mean_estimate_22(10) - H_22, computed=True)
with checked("M06.22c"):
    wbm.check("M06.22c", mean_estimate_22(100) - H_22, computed=True)
with checked("M06.22d"):
    wbm.check("M06.22d", float(np.mean([plugin_entropy(sample, 6) < H_22 for sample in SAMPLES_22[10]])), computed=True)
with checked("M06.22e"):
    wbm.check("M06.22e", mean_estimate_22(10) + (6 - 1) / (2 * 10 * math.log(2)) - H_22, computed=True)

**Le biais pour les cinq tailles** (cette cellule utilise ta fonction).

In [ ]:
with checked("M06.22 tableau"):
    print("    n · mean estimate ·    bias · −(k − 1)/(2n ln 2) · bias after Miller-Madow")
    for n_22 in SAMPLES_22:
        mean_22 = float(np.mean([plugin_entropy(sample, 6) for sample in SAMPLES_22[n_22]]))
        theory_22 = -(6 - 1) / (2 * n_22 * math.log(2))
        print(f"{n_22:5d} · {mean_22:13.4f} · {mean_22 - H_22:7.4f} · {theory_22:18.4f} · {mean_22 - theory_22 - H_22:10.4f}")

### Ex M06.23 — Un diagramme de l'information : H(X), H(Y), H(X | Y), I(X ; Y) 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner les entropies d'un couple de variables en barres alignées, et y lire les règles de la chaîne et l'information mutuelle.  
**Prérequis :** fiche §3 et §4 ; R0 (matplotlib) · **Parcours :** C

Le diagramme de la fiche (§4) se dessine aussi en barres horizontales, à la même échelle :
- une barre « H(X) », faite de deux morceaux : $H(X \mid Y)$ puis $I(X ; Y)$ ;
- une barre « H(Y) », faite de $I(X ; Y)$ puis $H(Y \mid X)$, décalée pour que les deux morceaux $I(X ; Y)$ soient exactement l'un au-dessus de l'autre ;
- une barre « H(X, Y) », de 0 à $H(X, Y)$ (en un morceau, ou en trois).

Écris `draw_information_bars(ax, joint)`, qui dessine ces barres dans l'axe `ax` (`ax.barh(position, largeur, left=début)`), en bits, avec le nom de chaque barre sur l'axe vertical (`ax.set_yticks(positions, labels=[...])`, avec les noms `H(X)`, `H(Y)` et `H(X, Y)`) et le mot « bits » dans le nom de l'axe horizontal. Calcule les entropies avec tes fonctions de `mymath.info` (M06.16 et M06.17) ou avec NumPy. Puis dessine le diagramme du tableau `T23 = [[0.3, 0.05, 0.05], [0.1, 0.2, 0.3]]`.

Dans ta copie : lis sur ta figure les deux règles de la chaîne et les trois écritures de $I(X ; Y)$. À quoi ressemble la figure pour deux variables indépendantes ? Et si $Y = X$ ?

In [ ]:
def draw_information_bars(ax, joint):
    """In ax, three horizontal bars, in bits: H(X) = H(X | Y) + I(X ; Y), H(Y) = I(X ; Y) + H(Y | X) shifted so that
    the two I(X ; Y) pieces are aligned, and H(X, Y); the names of the bars on the vertical axis."""
    raise NotImplementedError("les entropies, puis ax.barh(..., left=...), ax.set_yticks et ax.set_xlabel")

In [ ]:
with checked("M06.23 figure"):
    fig, ax = plt.subplots(figsize=(7, 3))
    try:
        created = drawn(draw_information_bars, ax, [[0.3, 0.05, 0.05], [0.1, 0.2, 0.3]])
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M06.23 : draw_information_bars ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans "
              "l'axe reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur un autre exemple ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M06.23 vérifications"):
    from matplotlib.patches import Rectangle
    problems = []
    joint_t = np.array([[0.25, 0.05], [0.15, 0.15], [0.05, 0.35]])     # another example


    def entropy_t(p):
        p = np.asarray(p, dtype=float).ravel()
        p = p[p > 0]
        return float(-np.sum(p * np.log2(p)))


    hx, hy, hxy = entropy_t(joint_t.sum(axis=1)), entropy_t(joint_t.sum(axis=0)), entropy_t(joint_t)
    fig, ax = plt.subplots(figsize=(7, 3))
    try:
        created = drawn(draw_information_bars, ax, joint_t)
        fig.canvas.draw()
        labels = " ".join(t.get_text() for t in [*ax.get_yticklabels(), *ax.texts]).replace(" ", "")
        xlabel = ax.get_xlabel()
        rows = {}
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and abs(patch.get_width()) > 1e-9:
                key = round(patch.get_y() + patch.get_height() / 2, 6)
                rows.setdefault(key, []).append((patch.get_x(), patch.get_x() + patch.get_width()))
    finally:
        plt.close(fig)                                                     # only the checks below are shown here
    rows = [sorted(row) for row in rows.values()]

    def has_row(*intervals):
        return any(len(row) == len(intervals) and all(abs(a - c) < 1e-6 and abs(b - d) < 1e-6
                                                      for (a, b), (c, d) in zip(row, intervals)) for row in rows)

    if created:
        problems.append("draw_information_bars ouvre sa propre figure : dessine dans l'axe reçu")
    elif not rows:
        problems.append("il manque les barres horizontales (ax.barh)")
    else:
        if not has_row((0, hxy - hy), (hxy - hy, hx)):
            problems.append("la barre de H(X) a deux morceaux : H(X | Y), de 0 à H(X | Y), puis I(X ; Y), jusqu'à H(X)")
        if not has_row((hxy - hy, hx), (hx, hxy)):
            problems.append("la barre de H(Y) commence à H(X | Y), sous le morceau I(X ; Y) de la barre de H(X), puis "
                            "continue par H(Y | X)")
        if not (has_row((0, hxy)) or has_row((0, hxy - hy), (hxy - hy, hx), (hx, hxy))):
            problems.append("la barre de H(X, Y) va de 0 à H(X, Y)")
        if not all(name in labels for name in ("H(X)", "H(Y)", "H(X,Y)")):
            problems.append("écris le nom de chaque barre sur l'axe vertical : H(X), H(Y), H(X, Y)")
        if "bit" not in xlabel:
            problems.append("l'axe horizontal est en bits : écris-le (ax.set_xlabel)")
    if problems:
        print("❌ Ex M06.23 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M06.23 : les trois barres sont justes et alignées, avec leurs noms et leur unité.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §2, §5 et §7. Trois erreurs à trouver, 0 log 0 calculé proprement, puis un nombre à deviner en un minimum de questions.

### Ex M06.24 — Chasse à l'erreur : log en base 10, 0 log 0, KL à l'envers 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur l'entropie, la cross-entropy et la divergence KL, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1, §2, §5 et §8 · **Parcours :** C

Des données suivent la loi `p`, un modèle la loi `q`. Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle veut calculer ; chaque ligne utilise les variables des lignes précédentes telles qu'elles **devraient** être (on suppose les lignes précédentes justes). Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
p = np.array([0.5, 0.375, 0.125, 0.0])        # the law of the data
q = np.array([0.25, 0.5, 0.125, 0.125])       # the law of the model
H_p = -np.sum(p * np.log2(p))                 # (A) the entropy of p, in bits
H_p_nats = H_p * math.log(2)                  # (B) the same entropy, in nats
H_pq = -np.sum(p * np.log10(q))               # (C) the cross-entropy H(p, q), in bits
KL = np.sum(q * np.log2(q / p))               # (D) the divergence KL(p ‖ q), in bits
info_4 = -math.log2(q[3])                     # (E) the information of the 4th outcome under the model, in bits
perplexity = 2 ** H_pq                        # (F) the perplexity of the model on the data, 2 ** H(p, q)
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, que renvoie-t-elle (un nombre faux, `nan` ou `inf`), et comment la corriger ? Pourquoi le résultat d'une ligne fausse peut-il passer inaperçu ?

In [ ]:
answer_M06_24a = ...  # a) letters, as text
answer_M06_24b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M06.24a", answer_M06_24a)
wbm.check("M06.24b", answer_M06_24b)

### Ex M06.25 — Calculer 0 log 0 proprement (`scipy.special.xlogy`) 🛠️ ★ ⏱️ 10 min
**Objectif :** calculer une entropie sans `nan` ni avertissement quand des probabilités sont nulles, avec la fonction de SciPy faite pour cela, sur un vecteur comme sur une loi jointe.  
**Prérequis :** fiche §2 ; Ex M06.24 · **Parcours :** C

`p * np.log2(p)` vaut `nan` dès qu'une probabilité est nulle ($0 \times (-\infty)$), avec un avertissement. Le filtre `p[p > 0]` règle le problème, mais il aplatit le tableau, ce qui gêne quand on veut garder sa forme (pour une somme par ligne, par exemple). SciPy a une fonction faite pour cela : `scipy.special.xlogy(x, y)` calcule $x \ln y$ case par case, et renvoie 0 quand $x = 0$, même si $y = 0$, sans avertissement.

Écris `entropy_xlogy(p, base=2)`, l'entropie de la loi `p` (un vecteur, ou un tableau à deux dimensions : une loi jointe) avec `xlogy`, sans filtre ni boucle, en `float`. La vérification compare ta fonction à un calcul de référence sur plusieurs lois, dont des lois jointes et des probabilités nulles, et vérifie qu'elle n'émet aucun avertissement ; puis elle calcule `entropy_xlogy(P25)` pour `P25 = [0, 0.05, 0.55, 0, 0.4]` (4 décimales).

Dans ta copie : pourquoi la convention $0 \log 0 = 0$ est-elle la bonne (quelle est la limite de $x \ln x$ quand $x$ tend vers 0 ?) ? Que calcule `scipy.special.entr(x)`, une autre fonction de SciPy ? Comment obtiendrais-tu, avec `xlogy` et `axis=1`, l'entropie de chaque ligne d'un tableau ?

In [ ]:
from scipy.special import xlogy


def entropy_xlogy(p, base=2):
    """The entropy of the law p (a vector, or a 2-D joint law) in the given base, with xlogy: no filter, no loop."""
    raise NotImplementedError("-xlogy(p, p), sa somme, divisée par ln(base), en float")

In [ ]:
with checked("M06.25 vérifications"):
    import warnings
    problems = []
    laws_t = [[0.5, 0.5], [0.0, 1.0], [0.2, 0.0, 0.3, 0.5], [[0.1, 0.0], [0.35, 0.55]], [[0.25, 0.25], [0.25, 0.25]]]
    for law in laws_t:
        array_t = np.asarray(law, dtype=float)
        positive_t = array_t[array_t > 0]
        expected = float(-np.sum(positive_t * np.log2(positive_t)))
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            got = entropy_xlogy(law)
        if got is None:
            problems.append("entropy_xlogy ne renvoie rien : il manque return")
            break
        if caught:
            problems.append(f"entropy_xlogy({law}) émet un avertissement ({caught[0].category.__name__}) : xlogy n'en "
                            "émet pas, même pour une probabilité nulle")
            break
        if not isinstance(got, float):
            problems.append(f"renvoie un float de Python (float(...)), pas un {type(got).__name__}")
            break
        if not math.isclose(got, expected, abs_tol=1e-12):
            problems.append(f"entropy_xlogy({law}) devrait valoir {expected:.6f} bit(s), pas {got:.6f}"
                            + (" : la somme porte sur toutes les cases du tableau" if np.ndim(law) == 2 else ""))
            break
    if not problems:
        nats_t = entropy_xlogy([0.2, 0.3, 0.5], base=math.e)
        if not math.isclose(nats_t, -sum(v * math.log(v) for v in (0.2, 0.3, 0.5)), abs_tol=1e-12):
            problems.append("avec base=math.e, le résultat est en nats : divise par ln(base)")
    if problems:
        print("❌ Ex M06.25 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M06.25 : entropy_xlogy est juste, pour des vecteurs et des lois jointes, sans aucun avertissement.")
with checked("M06.25"):
    wbm.check("M06.25", entropy_xlogy([0, 0.05, 0.55, 0, 0.4]), computed=True)

### Ex M06.26 — Défi : deviner un nombre de 1 à 100 en un minimum de questions 🏆 ★★★ ⏱️ 25 min
**Objectif :** concevoir une stratégie de questions par oui ou non qui approche l'entropie, pour une loi uniforme et pour une loi inégale.  
**Prérequis :** fiche §2 et §7 · **Parcours :** C

Je pense à un nombre entre 1 et 100, et tu poses des questions de la forme « ton nombre est-il inférieur ou égal à $t$ ? », auxquelles je réponds par oui ou par non. Chaque suite de réponses est un mot d'un code préfixe binaire (fiche §7) : en moyenne, il faut donc au moins $H$ questions, où $H$ est l'entropie de la loi du nombre, et l'on peut faire moins de $H + 1$… à condition de bien choisir ses questions.

Écris `guess_number(is_at_most, probs)`, qui trouve le nombre en appelant `is_at_most(t)` (qui renvoie `True` si le nombre est inférieur ou égal à `t`), et le renvoie. `probs[k]` est la probabilité du nombre `k + 1` : ta stratégie peut s'en servir. **Les objectifs** :
- pour la loi uniforme sur 1, …, 100 : au plus 7 questions pour chaque nombre, et au plus 6,72 en moyenne ;
- pour la loi où la probabilité du nombre $n$ est proportionnelle à $\frac{1}{n}$ (les petits nombres sont les plus probables) : au plus 5,40 questions en moyenne, pondérée par les probabilités.

La vérification essaie ta fonction sur les 100 nombres, pour les deux lois, en comptant les questions. Elle vérifie aussi `bound_26`, l'entropie en bits de la seconde loi, que tu calcules (4 décimales) : le nombre moyen de questions ne peut pas descendre en dessous.

Dans ta copie : pourquoi 6,72 est-il le mieux possible pour la loi uniforme (combien de nombres peut-on distinguer avec 6 questions ? avec 7 ?) ? Pourquoi couper l'intervalle en deux moitiés de même longueur ne suffit-il pas pour la seconde loi ? Compare ton score à `bound_26`.

In [ ]:
PROBS_UNIFORM_26 = np.full(100, 0.01)                      # probs[k]: the probability of the number k + 1
PROBS_HARMONIC_26 = 1 / np.arange(1, 101)
PROBS_HARMONIC_26 = PROBS_HARMONIC_26 / PROBS_HARMONIC_26.sum()   # the probability of n is proportional to 1/n

In [ ]:
def guess_number(is_at_most, probs):
    """Find the number (1 to len(probs)) with questions is_at_most(t) -> bool; probs[k] is the probability of k + 1."""
    raise NotImplementedError("garde l'intervalle des nombres possibles, et coupe-le en deux parts bien choisies")


bound_26 = ...  # the entropy, in bits, of PROBS_HARMONIC_26 (4 decimals)

In [ ]:
with checked("M06.26 défi"):
    problems, results = [], {}
    for law_name, probs_t in [("uniforme", PROBS_UNIFORM_26), ("en 1/n", PROBS_HARMONIC_26)]:
        counts_t = []
        for secret in range(1, 101):
            questions = []

            def is_at_most(t, secret=secret, questions=questions):
                questions.append(t)
                if len(questions) > 200:
                    raise RuntimeError("plus de 200 questions : ta stratégie tourne en rond")
                return secret <= t

            try:
                got = guess_number(is_at_most, list(probs_t))
            except NotImplementedError:
                raise
            except Exception as exc:  # noqa: BLE001
                problems.append(f"loi {law_name}, nombre {secret} : guess_number lève {type(exc).__name__} ({exc})")
                break
            if got != secret:
                problems.append(f"loi {law_name} : pour le nombre {secret}, guess_number renvoie {got!r}")
                break
            counts_t.append(len(questions))
        if problems:
            break
        results[law_name] = (max(counts_t), float(np.dot(probs_t, counts_t)))
    if problems:
        print("❌ Ex M06.26 : " + problems[0] + ".")
    else:
        worst_u, mean_u = results["uniforme"]
        mean_h = results["en 1/n"][1]
        print(f"loi uniforme : {worst_u} questions au plus, {mean_u:.3f} en moyenne · loi en 1/n : {mean_h:.3f} en "
              "moyenne")
        failures = []
        if worst_u > 7:
            failures.append("plus de 7 questions pour un des nombres, avec la loi uniforme")
        if mean_u > 6.72 + 1e-9:
            failures.append("plus de 6,72 questions en moyenne avec la loi uniforme")
        if mean_h > 5.40 + 1e-9:
            failures.append("plus de 5,40 questions en moyenne avec la loi en 1/n : sers-toi des probabilités")
        if failures:
            print("❌ Ex M06.26 : objectif pas encore atteint : " + " ; ".join(failures) + ".")
        else:
            print("✅ Ex M06.26 : défi réussi, les deux objectifs sont atteints.")
with checked("M06.26"):
    wbm.check("M06.26", bound_26)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer l'information d'un événement et l'entropie d'une loi, en bits et en nats ?
2. Sais-tu calculer des entropies conjointe et conditionnelle et une information mutuelle à partir d'un tableau, et dire ce que mesure chacune ?
3. Sais-tu calculer une cross-entropy et une divergence KL, les lire en bits perdus, et dire pourquoi la KL est positive (Jensen) ?
4. Sais-tu dire si des longueurs de code sont possibles (Kraft), et comparer la longueur moyenne d'un code à l'entropie ?

**La suite** : le chapitre 6 de workbookIA (entropie, cross-entropy, KL, codes), puis CP1, l'examen blanc de la partie I.